## Import Modules

In [1]:
# Make sure you are the parent directory
from pathlib import Path
import sys

# Define directory path: notebooks/ -> project root
PROJECT_ROOT = Path.cwd().resolve().parent
display(PROJECT_ROOT)

# Add it to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PosixPath('/expanse/lustre/projects/uci157/bguo3/DSC_288R')

In [ ]:
# PySpark libs
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType, IntegerType,
    FloatType, StringType, BooleanType
)

# Other libs
import kagglehub
from src.utils.pyspark_helper import create_spark_session, memory_count

## Display General Info

## Set up for Spark

In [2]:
# Set up for Spark app & resource allocation
spark = create_spark_session("steam_reviews_game_review_data_comparison")

## Data Ingestion

In [3]:
# Fetch data through Kaggle API
BASE_PATH = kagglehub.dataset_download(
    "artermiloff/steam-games-dataset",
    force_download=True,
)
print("Downloaded to:", BASE_PATH)

100%|██████████| 418M/418M [00:04<00:00, 98.1MB/s] 

Extracting files...


Downloaded to: /home/bguo3/.cache/kagglehub/datasets/artermiloff/steam-games-dataset/versions/2


In [6]:
# Define Spark DF schema
schema = { "appid": IntegerType() }

cols = list(schema.keys())

In [10]:
# Load Spark DF through CSV
CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}
CSV_PATH = f'file:{BASE_PATH}/games_may2024_cleaned.csv'

df1 = (
    spark.read
    .options(**CSV_OPTIONS)
    .csv(CSV_PATH)
    .select([
        F.col(c).cast(c_type).alias(c)
        for c, c_type in schema.items()
    ])
)

CSV_PATH = f'file:{BASE_PATH}/games_march2025_cleaned.csv'

df2 = (
    spark.read
    .options(**CSV_OPTIONS)
    .csv(CSV_PATH)
    .select([
        F.col(c).cast(c_type).alias(c)
        for c, c_type in schema.items()
    ])
)

In [14]:
distinct_app_set = {
    row["appid"]
    for row in (
        df1.unionByName(df2)
           .select("appid")
           .where(F.col("appid").isNotNull())
           .distinct()
           .collect()
    )
}
len(distinct_app_set)

98528

In [16]:
distinct_app1 = set(df1.select("appid")
   .where(F.col("appid").isNotNull())
   .distinct()
   .collect())
len(distinct_app1)

83646

In [17]:
distinct_app2 = set(df2.select("appid")
   .where(F.col("appid").isNotNull())
   .distinct()
   .collect())
len(distinct_app2)

89618

In [19]:
EXPANSE_ROOT = "/expanse/lustre/scratch/bguo3/temp_project"
FULL_PARQUET_PATH = f"file:{EXPANSE_ROOT}/steam_reviews/full_parquet"

df_full = spark.read.parquet(FULL_PARQUET_PATH)
print(f"Read full parquet successfully from: {FULL_PARQUET_PATH}")

# Ensure the full data is loaded with all columns intact
df_full.printSchema()

Read full parquet successfully from: file:/expanse/lustre/scratch/bguo3/temp_project/steam_reviews/full_parquet
root
 |-- author_steamid: long (nullable = true)
 |-- appid: integer (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)



In [37]:
distinct_app_full = {
    row["appid"]
    for row in (
        df_full
        .select("appid")
        .where(F.col("appid").isNotNull())
        .distinct()
        .collect()
    )
}
len(distinct_app_full)

106478

In [39]:
len(distinct_app_full - distinct_app_set)

35504

In [40]:
len(distinct_app_set - distinct_app_full)

27554

In [45]:
explore_later = (distinct_app_set - distinct_app_full)
import pickle

with open("distinct_app_full.pkl", "wb") as f:
    pickle.dump(explore_later, f)

## Display General Info

In [9]:
# Verify to have a correct schema
df1.printSchema()

root
 |-- appid: integer (nullable = true)



In [ ]:
# Estimate the size of full dataset after columns removed
memory_count(df_full)

Total row counts: 113885601 rows
Total estimated size: 48.32 GB


In [14]:
spark.stop()